In [1]:
print("hello python")

hello python


### Data Loading and Preprocessing

In [2]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

In [3]:
print(torch.cuda.is_available())       # True?
print(torch.cuda.get_device_name(0))   # Should show RTX 4060

True
NVIDIA GeForce RTX 4060 Laptop GPU


In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)  # should print: cuda

cuda


In [5]:
os.listdir()

['.ipynb_checkpoints', 'Untitled.ipynb']

In [6]:
data = pd.read_csv("../data/league_of_legends_data_large.csv")
data.head()

,win,kills,deaths,assists,gold_earned,cs,wards_placed,wards_killed,damage_dealt
0,0,16,6,19,17088,231,11,7,15367
1,1,8,8,5,14865,259,10,2,38332
2,0,0,17,11,15919,169,14,5,24642
3,0,19,11,1,11534,264,14,3,15789
4,0,12,7,6,18926,124,15,7,40268


In [7]:
#Separate win (target) and the remaining columns (features).
X = data.drop('win',axis=1)
y = data['win']

X_train , X_test , y_train , y_test = train_test_split(
    X,y,
    test_size=0.2,
    random_state=42
) 

In [8]:
scaler = StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.fit_transform(X_test)

In [9]:
input_dim = X_train.shape[1]
input_dim

8

In [10]:
X_train.dtype

dtype('float64')

In [11]:
y_test.dtype

dtype('int64')

In [12]:
#Because PyTorch models use float32 by default, and StandardScaler outputs float64
#.values convert pandas series into numpy array
X_train = torch.tensor(X_train,dtype=torch.float32)
y_train = torch.tensor(y_train.values,dtype=torch.float32)
X_test = torch.tensor(X_test,dtype=torch.float32)
y_test = torch.tensor(y_test.values,dtype=torch.float32)

In [13]:
X_train.dtype

torch.float32

In [14]:
y_test.dtype

torch.float32

In [15]:
X_train[0].shape

torch.Size([8])

### Implement a logistic regression model using PyTorch


In [16]:
class logistic_regression(nn.Module):
    def __init__(self,in_dim):
        super(logistic_regression,self).__init__()
        self.linear = nn.Linear(in_dim,1)
    def forward(self,x):
        return torch.sigmoid(self.linear(x))        

In [17]:
criterion = nn.BCELoss()

In [18]:
# X_train.shape[0] → 800   (samples)
# X_train.shape[1] → 8     (features)
model = logistic_regression(input_dim)

In [19]:
optimizer=optim.SGD(model.parameters(),lr=0.01)

### Train the logistic regression model

In [20]:
epochs = 1000
y_train = y_train.view(-1, 1) 
data = TensorDataset(X_train,y_train)
trainloader = DataLoader(dataset = data, batch_size = 1)

In [21]:
for epoch in range(epochs):
    for x,y in trainloader:
        yhat = model(x)
        loss = criterion(yhat,y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    if(epoch+1) % 100 == 0 :
            print(f"Epoch {epoch+1} , Loss: {loss.item():.4f}")

Epoch 100 , Loss: 0.9703
Epoch 200 , Loss: 0.9703
Epoch 300 , Loss: 0.9703
Epoch 400 , Loss: 0.9703
Epoch 500 , Loss: 0.9703
Epoch 600 , Loss: 0.9703
Epoch 700 , Loss: 0.9703
Epoch 800 , Loss: 0.9703
Epoch 900 , Loss: 0.9703
Epoch 1000 , Loss: 0.9703


In [24]:
model.eval()

with torch.no_grad():
    yhat_train = model(X_train)
    yhat_test = model(X_test)

In [25]:
train_acc = (yhat_train >=0.5).float().eq(y_train).float().mean()
test_acc = (yhat_test >=0.5).float().eq(y_test.view(-1,1)).float().mean()

In [26]:
print(f"Train Accuracy: {train_acc.item():.4f}")
print(f"Test Accuracy: {test_acc.item():.4f}")

Train Accuracy: 0.0000
Test Accuracy: 0.0000


In [27]:
print(y_train.unique())   # kya 0 aur 1 dono hain?
print(y_train.dtype)      # float32 hona chahiye
print(X_train.mean(), X_train.std())  # normalized hai?

tensor([0.3247, 0.3319, 0.3496, 0.3589, 0.3625, 0.3644, 0.3666, 0.3690, 0.3708,
        0.3709, 0.3710, 0.3727, 0.3738, 0.3742, 0.3750, 0.3751, 0.3757, 0.3759,
        0.3794, 0.3796, 0.3806, 0.3812, 0.3821, 0.3823, 0.3842, 0.3856, 0.3856,
        0.3865, 0.3867, 0.3872, 0.3873, 0.3888, 0.3901, 0.3906, 0.3909, 0.3923,
        0.3936, 0.3951, 0.3952, 0.3959, 0.3966, 0.3970, 0.3990, 0.3990, 0.3990,
        0.3995, 0.3998, 0.4001, 0.4005, 0.4007, 0.4017, 0.4020, 0.4020, 0.4023,
        0.4028, 0.4036, 0.4037, 0.4041, 0.4051, 0.4059, 0.4066, 0.4066, 0.4072,
        0.4077, 0.4086, 0.4088, 0.4089, 0.4090, 0.4101, 0.4113, 0.4123, 0.4126,
        0.4127, 0.4127, 0.4129, 0.4135, 0.4138, 0.4140, 0.4142, 0.4143, 0.4147,
        0.4149, 0.4160, 0.4166, 0.4171, 0.4174, 0.4182, 0.4183, 0.4188, 0.4190,
        0.4202, 0.4207, 0.4209, 0.4211, 0.4215, 0.4216, 0.4217, 0.4220, 0.4221,
        0.4227, 0.4229, 0.4238, 0.4238, 0.4239, 0.4241, 0.4243, 0.4255, 0.4258,
        0.4263, 0.4264, 0.4267, 0.4270, 